In [1]:
import os
import csv
from osgeo import gdal
import numpy as np
import re
import itertools
import pandas as pd
import rasterio
import geopandas as gpd
import sys
from shapely.geometry import Polygon
import math
import gc
from sqlalchemy import create_engine, text
import rasterio.mask
import dask.array as da

In [2]:
def get_raster_file_list(path):
    """Get a list of the raster files inside the folder"""
    File_list = [] #f for f in os.listdir(path) if os.isfile(mypath,f)
    for file in os.listdir(path):
        if file.endswith(".tif") or file.endswith(".tiff"):
            if file not in File_list:
                File_list.append(os.path.join(path,file))
        else:
            pass
    return File_list

def read_csv_first_column(csv_path):
    """
    Reads a CSV file and extracts the first column as landcover class values.
    Skips the header row.

    Returns: list of integers
    """
    try:
        classes = []
        with open(csv_path, "r", newline="", encoding="utf-8") as f:
            reader = csv.reader(f)
            next(reader)  # skip header
            for row in reader:
                if row and row[0].strip() != "":
                    try:
                        classes.append(int(row[0]))
                    except ValueError:
                        pass  # ignore non-numeric values
        return classes
    except:
        return None

def match_rasters_by_country(data_rasters, landcover_rasters):
    # Extract country = first token before underscore
    def country(f): 
        return os.path.basename(f).split("_")[0]

    # Build lookup tables
    data_dict = {}
    for f in data_rasters:
        data_dict.setdefault(country(f), []).append(f)

    lc_dict = {}
    for f in landcover_rasters:
        lc_dict.setdefault(country(f), []).append(f)

    # Match only countries appearing in both lists
    matched_pairs = []
    for c in set(data_dict) & set(lc_dict):
        for d, l in itertools.product(data_dict[c], lc_dict[c]):
            matched_pairs.append((d, l))

    return matched_pairs

def match_rasters_by_country_year(data_rasters, landcover_rasters):
    """
    Matches data rasters and landcover rasters using:
        - the first string in the filename (country)
        - the last 4-digit number in the filename (year)

    Expected filename examples:
        BRA_carbon_2000.tif
        BRA_landcover_2000.tif
        USA_vsc_2015_v1.tif
        USA_landcover_2015.tif

    Returns:
        list of (data_raster, landcover_raster) pairs
    """

    def extract_country_and_year(filepath):
        fname = os.path.basename(filepath)

        # country = first token before first underscore
        country = fname.split("_")[0]

        # year = last 4 consecutive digits before non-digit/end
        year_match = re.search(r"(\d{4})(?=\D*$)", fname)
        year = year_match.group(1) if year_match else None

        return country, year


    def build_dict(raster_list):
        """
        Build dict indexed by (country, year):
            { (country, year) : filepath }
        """
        d = {}
        for r in raster_list:
            country, year = extract_country_and_year(r)
            if country and year:
                d.setdefault((country, year), []).append(r)   # append many matches
        return d


    # Build lookup tables
    data_dict = build_dict(data_rasters)
    lc_dict = build_dict(landcover_rasters)

    # Find matching (country, year) keys
    common_keys = sorted(set(data_dict.keys()) & set(lc_dict.keys()))

    # Prepare output pairs
    # matched_pairs = [(data_dict[key], lc_dict[key]) for key in common_keys]
    
    matched_pairs = []

    for key in common_keys:
        for data_file, lc_file in itertools.product(data_dict[key], lc_dict[key]):
            matched_pairs.append((data_file, lc_file))   # <- flat tuple

    return matched_pairs


def align_raster_to_reference(ref_raster, input_raster, output_raster):
    """
    Resamples/reprojects input_raster to match CRS, resolution, extent of ref_raster.
    Uses nearest neighbor resampling for categorical data.
    """
    if not os.path.exists(os.path.dirname(output_raster)):
        os.makedirs(os.path.dirname(output_raster), exist_ok=True)

    ref_ds = gdal.Open(ref_raster)
    if ref_ds is None:
        raise FileNotFoundError(f"Reference raster not found: {ref_raster}")

    options = gdal.WarpOptions(
        format='GTiff',
        dstSRS=ref_ds.GetProjection(),
        width=ref_ds.RasterXSize,
        height=ref_ds.RasterYSize,
        resampleAlg='nearest', # It is a landcover dataset, so nearest neighbor resampling is appropriate
        outputBounds=[
            ref_ds.GetGeoTransform()[0],  # minX
            ref_ds.GetGeoTransform()[3] + ref_ds.RasterYSize * ref_ds.GetGeoTransform()[5],  # minY
            ref_ds.GetGeoTransform()[0] + ref_ds.RasterXSize * ref_ds.GetGeoTransform()[1],  # maxX
            ref_ds.GetGeoTransform()[3]  # maxY
        ],
        # Lets see if this works
        creationOptions=[
            "COMPRESS=DEFLATE",  
            "TILED=YES"          
    ]
    )

    result = gdal.Warp(destNameOrDestDS=output_raster,
                       srcDSOrSrcDSTab=input_raster,
                       options=options)

    if result is None:
        raise RuntimeError(f"gdal.Warp failed to generate: {output_raster}")

    result = None
    return output_raster

def landcover_mask_rasters(data_rasters, landcover_rasters, class_values, output_dir):
    """
    Masks each data raster by landcover classes.
    Automatically aligns landcover raster if shapes differ.
    """
    # Create the output dir
    os.makedirs(output_dir, exist_ok=True)

    # Match rasters by country + year
    # pairs = match_rasters_by_country_year(data_rasters, landcover_rasters)
    pairs = match_rasters_by_country(data_rasters, landcover_rasters)

    if not pairs:
        print("No matching raster pairs found. Check filenames!")
        return

    for data_file, lc_file in pairs[:]: #ACHTUNG: Update this part
        print(f"\nProcessing pair:\n  DATA: {data_file}\n  LC  : {lc_file}")

        # Open data raster
        data_ds = gdal.Open(data_file)
        if data_ds is None:
            print(f"Cannot open data raster: {data_file}, skipping...")
            continue
        
        band = data_ds.GetRasterBand(1)
        data_arr = band.ReadAsArray()

        # original nodata
        nodata_value = band.GetNoDataValue()

        # Getnodatvalue requires always something
        if nodata_value is None:
            nodata_value = 0

        # Get the raster properties        
        geotrans = data_ds.GetGeoTransform()
        proj = data_ds.GetProjection()
        xsize = data_ds.RasterXSize
        ysize = data_ds.RasterYSize
        base_name = os.path.splitext(os.path.basename(data_file))[0]

        # Open landcover raster
        lc_ds = gdal.Open(lc_file)
        if lc_ds is None:
            print(f"Cannot open landcover raster: {lc_file}, skipping...")
            continue
        lc_arr = lc_ds.GetRasterBand(1).ReadAsArray()

        # Check shapes of raster and lc if an alignment is needed
        if lc_arr.shape != data_arr.shape:
            print("  - Landcover raster shape mismatch. Aligning...")
            aligned_lc_path = os.path.join(output_dir, "tmp_aligned_lc.tif")
            
            # The first variable is the reference
            align_raster_to_reference(data_file, lc_file, aligned_lc_path)
            
            lc_ds = gdal.Open(aligned_lc_path)
            lc_arr = lc_ds.GetRasterBand(1).ReadAsArray()
            print("  - Alignment done.")
        
        # Determine landcover nodata
        lc_nodata = lc_ds.GetRasterBand(1).GetNoDataValue()
        
        # ------------------------------------------
        # CASE 1 — class_values is None → mask ALL presence
        # ------------------------------------------
        if class_values is None:
            print("Masking all landcover presence (class_values=None)")

            # mask = all valid lc pixels (anything that is not nodata)
            mask = (lc_arr != lc_nodata)
            out_arr = np.where(mask, data_arr, nodata_value)
            # Use the lc name end.

            lc_name = os.path.splitext(os.path.basename(lc_file))[0]
            out_name = f"{base_name}_{lc_name.split('_')[-1]}.tif" # Get the second last part

            
            out_path = os.path.join(output_dir, out_name)

            driver = gdal.GetDriverByName("GTiff")
            out_ds = driver.Create(out_path, xsize, ysize, 1, gdal.GDT_Float32,
                                options=["COMPRESS=DEFLATE", "TILED=YES"])
            out_ds.SetGeoTransform(geotrans)
            out_ds.SetProjection(proj)
            band = out_ds.GetRasterBand(1)
            band.WriteArray(out_arr)
            band.SetNoDataValue(nodata_value)
            band.FlushCache()
            out_ds = None

        # ------------------------------------------
        # CASE 2 — class_values provided → existing behavior
        # ------------------------------------------
        else:    
            # Apply mask per class
            for lc_val in class_values:
                print(f"  - Masking class {lc_val}", end="")
                mask = (lc_arr == lc_val)
                out_arr = np.where(mask, data_arr, nodata_value)

                out_name = f"{base_name}_{lc_val}.tif"
                out_path = os.path.join(output_dir, out_name)

                driver = gdal.GetDriverByName("GTiff")
                out_ds = driver.Create(out_path, xsize, ysize, 1, gdal.GDT_Float32,
                                    options=["COMPRESS=DEFLATE", "TILED=YES"])
                out_ds.SetGeoTransform(geotrans)
                out_ds.SetProjection(proj)
                band = out_ds.GetRasterBand(1)
                band.WriteArray(out_arr)
                band.SetNoDataValue(nodata_value)
                band.FlushCache()
                out_ds = None


            
        print(f"✓ Finished processing {base_name}")
        # Clean the variable
        # Close ALL GDAL datasets
        try:
            del lc_ds
            # delete temp file safely
            os.remove(aligned_lc_path)
        except: 
            pass

        
        
    print("\nAll rasters processed successfully.")



In [12]:
"""Inputs for classifier."""
landcover_classes_csv = r"Y:\z_resources\justus\im_nca_postprocessing_ungbf\colombia_landcover_classes.csv"
# landcover_classes_csv = None

raster_files_path = r"Y:\z_resources\justus\im_nca_postprocessing_ungbf\02_klab_main_rasters\00import"

landcover_files_path = r"Y:\z_resources\justus\im_nca_postprocessing_ungbf\01_mask_rasters\01_current_processing"

output_dir = r"Y:\z_resources\justus\im_nca_postprocessing_ungbf\03_masked_outputs\uganda_test"

In [10]:
raster_files_list = get_raster_file_list(raster_files_path)
landcover_files_list = get_raster_file_list(landcover_files_path)

lc_classes_list = read_csv_first_column(landcover_classes_csv)

In [ ]:
country_pairs = match_rasters_by_country(raster_files_list, landcover_files_list)
country_pairs[:]

In [ ]:
file_pairs = match_rasters_by_country_year(raster_files_list, landcover_files_list)
file_pairs[:]

In [ ]:
# classes = read_landcover_classes(csv_path)
landcover_mask_rasters(raster_files_list, landcover_files_list, lc_classes_list, output_dir)

## For the wetlands and coral

In [ ]:
def align_raster_to_reference_2(input_raster, reference_raster, output_raster):
    """
    Align input_raster to the exact grid of reference_raster.
    Ensures shape, pixel size, extent, projection all match.
    """

    ref = gdal.Open(reference_raster)
    print("GeoTransform:", ref.GetGeoTransform())
    print("Size:", ref.RasterXSize, ref.RasterYSize)
    
    if ref is None:
        raise FileNotFoundError(f"Cannot open reference raster: {reference_raster}")

    ref_gt = ref.GetGeoTransform()
    ref_proj = ref.GetProjection()
    xsize = ref.RasterXSize
    ysize = ref.RasterYSize

    # Compute bounds from geotransform
    minx = ref_gt[0]
    maxy = ref_gt[3]
    maxx = minx + xsize * ref_gt[1]
    miny = maxy + ysize * ref_gt[5]

    warp_opts = gdal.WarpOptions(
        format="GTiff",
        dstSRS=ref_proj,
        outputBounds=(minx, miny, maxx, maxy),
        width=xsize,
        height=ysize,
        resampleAlg="nearest",
        creationOptions=[
            "COMPRESS=DEFLATE",
            "TILED=YES",
            "BLOCKXSIZE=256",
            "BLOCKYSIZE=256"
        ]
    )

    gdal.Warp(
        output_raster,
        input_raster,
        options=warp_opts
    )

    ref = None

def mask_raster_by_values(
    raster1_path,
    raster2_path,
    output_path,
    mask_values=(1, 6),
    nodata_value=0
):


    # --- Open reference raster2 ---
    r2 = gdal.Open(raster2_path)
    if r2 is None:
        raise FileNotFoundError(f"Cannot open raster2: {raster2_path}")

    r2_arr = r2.GetRasterBand(1).ReadAsArray()
    r2_shape = r2_arr.shape

    # --- Open raster1 ---
    r1 = gdal.Open(raster1_path)
    if r1 is None:
        raise FileNotFoundError(f"Cannot open raster1: {raster1_path}")

    r1_arr = r1.GetRasterBand(1).ReadAsArray()

    # =====================================================================
    # ALIGN IF SHAPES DO NOT MATCH
    # =====================================================================
    if r1_arr.shape != r2_shape:

        print("Shapes differ → aligning raster1 to raster2.")
        print(f"  raster1 shape: {r1_arr.shape}")
        print(f"  raster2 shape: {r2_shape}")

        tmp_aligned = raster1_path + "_aligned_tmp.tif"

        # run alignment (your function)
        align_raster_to_reference_2(raster1_path, raster2_path, tmp_aligned)

        print("The data is aligned")
        # reopen aligned raster
        r1_aligned = gdal.Open(tmp_aligned)
        if r1_aligned is None:
            raise RuntimeError("Alignment failed: Temporary aligned raster not created.")

        r1_arr = r1_aligned.GetRasterBand(1).ReadAsArray()

        # CLOSE the aligned dataset before deleting the file
        r1_aligned = None

        # strict validation
        if r1_arr.shape != r2_shape:
            os.remove(tmp_aligned)
            raise ValueError(
                f"Alignment FAILED: aligned raster1 has shape {r1_arr.shape}, "
                f"but raster2 has shape {r2_shape}"
            )

        # delete temp file safely
        os.remove(tmp_aligned)

    # =====================================================================
    # APPLY MASK
    # =====================================================================
    print("Applying mask...")
    mask = np.isin(r1_arr, mask_values)

    output_arr = r2_arr.copy()
    output_arr[mask] = nodata_value

    # =====================================================================
    # WRITE OUTPUT
    # =====================================================================
    print("Saving output...")
    driver = gdal.GetDriverByName("GTiff")
    out_ds = driver.Create(
        output_path,
        r2.RasterXSize,
        r2.RasterYSize,
        1,
        gdal.GDT_Float32,
        options=[
            "COMPRESS=DEFLATE",
            "TILED=YES",
            "BLOCKXSIZE=256",
            "BLOCKYSIZE=256"
        ]
    )

    out_ds.SetGeoTransform(r2.GetGeoTransform())
    out_ds.SetProjection(r2.GetProjection())

    out_band = out_ds.GetRasterBand(1)
    out_band.WriteArray(output_arr)
    out_band.SetNoDataValue(nodata_value)
    out_band.FlushCache()

    # close output
    out_ds = None

    print(f"Masked raster written to:\n  {output_path}")

mask_raster_by_values(
    raster1_path=r"Z:\z_resources\justus\im_nca_postprocessing_ungbf\landcover_rasters\indonesia_coral_systems_4326_2020.tif",
    raster2_path=r"Z:\z_resources\justus\im_nca_postprocessing_ungbf\landcover_rasters\indonesia_global_wetlands_20m_2020.tif",
    output_path=r"Z:\z_resources\justus\im_nca_postprocessing_ungbf\landcover_rasters\indonesia_global_wetlands_20m_2020_masked.tif",
    mask_values=(1, 6),
    nodata_value=0
)

## Vulnerability

In [2]:
def classify_vulnerability_rasters(
        raster_files,
        output_dir,
        nodata_value=-9999,
        use_alignment_function=False,
        align_function=None,
        reference_raster=None
    ):
    """
    Creates vulnerability classification rasters from a list of input rasters.

    Categories:
        LowVulnerability: < 33
        MediumVulnerability: 34–66
        HighVulnerability: 67–100
        UnknownVulnerability: nodata

    Output naming:
        originalname_LowVulnerability.tif
        originalname_MediumVulnerability.tif
        originalname_HighVulnerability.tif
        originalname_UnknownVulnerability.tif
    """

    os.makedirs(output_dir, exist_ok=True)

    for r in raster_files:
        print(f"Processing: {os.path.basename(r)}")

        # ------------------------------------------------------------------
        # Optional alignment step
        # ------------------------------------------------------------------
        if use_alignment_function and align_function and reference_raster:
            r_aligned = align_function(r, reference_raster)
            ds = gdal.Open(r_aligned)
            tmp_to_delete = r_aligned
        else:
            ds = gdal.Open(r)
            tmp_to_delete = None

        if ds is None:
            print(f"❌ ERROR: Cannot open {r}")
            continue

        band = ds.GetRasterBand(1)
        arr = band.ReadAsArray().astype(np.float32)

        if band.GetNoDataValue() is not None:
            nd = band.GetNoDataValue()
        else:
            nd = nodata_value

        # ------------------------------------------------------------------
        # Create category masks
        # ------------------------------------------------------------------
        low_mask     = (arr < 33) & (arr != nd)
        med_mask     = (arr >= 34) & (arr <= 66) & (arr != nd)
        high_mask    = (arr >= 67) & (arr <= 100) & (arr != nd)
        unknown_mask = (arr == nd)

        # ------------------------------------------------------------------
        # Prepare output arrays
        # ------------------------------------------------------------------
        low_arr     = np.where(low_mask, arr, nd)
        med_arr     = np.where(med_mask, arr, nd)
        high_arr    = np.where(high_mask, arr, nd)
        unknown_arr = np.where(unknown_mask, nd, nd)

        # ------------------------------------------------------------------
        # Save function
        # ------------------------------------------------------------------
        def save_output(array, suffix):
            out_path = os.path.join(
                output_dir,
                f"{os.path.splitext(os.path.basename(r))[0]}_{suffix}.tif"
            )
            driver = gdal.GetDriverByName("GTiff")
            out_ds = driver.Create(
                out_path,
                ds.RasterXSize,
                ds.RasterYSize,
                1,
                gdal.GDT_Float32,
                options=["COMPRESS=DEFLATE", "TILED=YES"]
            )
            out_ds.SetGeoTransform(ds.GetGeoTransform())
            out_ds.SetProjection(ds.GetProjection())
            out_ds.GetRasterBand(1).WriteArray(array)
            out_ds.GetRasterBand(1).SetNoDataValue(nd)
            out_ds = None
            print(f"  ✓ Saved: {os.path.basename(out_path)}")

        # ------------------------------------------------------------------
        # Write outputs
        # ------------------------------------------------------------------
        save_output(low_arr,     "LowVulnerability")
        save_output(med_arr,     "MediumVulnerability")
        save_output(high_arr,    "HighVulnerability")
        save_output(unknown_arr, "UnknownVulnerability")

        # ------------------------------------------------------------------
        # Cleanup
        # ------------------------------------------------------------------
        band = None
        ds = None
        del arr, low_arr, med_arr, high_arr, unknown_arr
        import gc; gc.collect()

        if tmp_to_delete and os.path.exists(tmp_to_delete):
            try:
                os.remove(tmp_to_delete)
            except PermissionError:
                print(f"⚠ Temp file still in use, cannot delete: {tmp_to_delete}")

    print("\nAll vulnerability rasters created successfully.")


In [17]:
raster_files_path = r"Z:\z_resources\justus\im_nca_postprocessing_ungbf\01_mask_rasters\02_current_processing"
raster_files_list = get_raster_file_list(raster_files_path)

output_dir = r"Z:\z_resources\justus\im_nca_postprocessing_ungbf\03_masked_outputs\vulnerability_files"


In [ ]:
classify_vulnerability_rasters(raster_files_list, output_dir)

## Aggregation

In [3]:
def load_vector_layer(db_name, user, password, host, port, table_name, schema='public', geom_col='geom'):
    """
    Connects to a PostGIS-enabled PostgreSQL database and loads a vector layer as a GeoDataFrame.
    
    Parameters:
    - db_name (str): Name of the PostgreSQL database.
    - user (str): Database username.
    - password (str): Database password.
    - host (str): Host address (e.g., 'localhost' or IP).
    - port (int): Port number (e.g., 5432).
    - table_name (str): Name of the table (vector layer) to load.
    - schema (str): Optional. Database schema containing the table (default is 'public').

    Returns:
    - gpd.GeoDataFrame: A GeoDataFrame containing the vector layer.
    """
    try:
        # Use pg8000 (pure Python driver)
        conn_str = f"postgresql+pg8000://{user}:{password}@{host}:{port}/{db_name}"
        engine = create_engine(conn_str)

        sql = text(f"SELECT * FROM {schema}.{table_name}")

        # Open a connection explicitly (SQLAlchemy 2.x requirement)
        with engine.connect() as conn:
            gdf = gpd.read_postgis(sql, conn, geom_col=geom_col)
        
        print(f"Successfully loaded {table_name} ({len(gdf)} features)")
        return gdf

    except Exception as e:
        print(f"Error loading vector layer: {e}")
        return None

def filter_countries(gdf_gadm, countries):

    available_countries = gdf_gadm['country_n'].unique().tolist()
    selected_countries = countries.copy()

    # Check for missing countries
    missing_countries = [c for c in selected_countries if c not in available_countries]

    if missing_countries:
        return print("Warning: These countries are not in the DataFrame:", missing_countries)
    
    else:
        gdf_gadm_countries = gdf_gadm[gdf_gadm['country_n'].isin(countries)]
        return gdf_gadm_countries

def get_geometry_grid(geodataframe, epsg):
    """
    get_geometry_grid creates a defined grid of the input territory extension.
    :geodataframe: the territory vector file ina gdf format.
    :epsg: the defined epsg of vector georeferenced data.
    :return: the grid as a geodataframe
    """
    
    #get the bounds of the territory
    xmin, ymin, xmax, ymax = geodataframe.total_bounds
    # define the size of the grid in degrees
    length = 5
    wide = 5
    # set the cols and rows
    cols = list(np.arange(xmin, xmax + wide, wide))
    rows = list(np.arange(ymin, ymax + length, length))
    # create a list of all the polygons containing the grid.
    polygons = []
    for x in cols[:-1]:
        for y in rows[:-1]:
            polygons.append(Polygon([(x,y), (x+wide, y), (x+wide, y+length), (x, y+length)]))
    # transform the polygon list into a Geoseries or Geodataframe.
    # grid = gpd.GeoSeries({'geometry':MultiPolygon(polygons)})
    grid = gpd.GeoDataFrame({'geometry':polygons}, crs=epsg)
    return grid

def area_of_pixel(pixel_size, center_lat):
    """
    area_of_pixel calculates the area, in hectares, of a wgs84 square raster
    tile given its latitude and side-length.
    This function is adapted from https://gis.stackexchange.com/a/288034.

    :param pixel_size: is the length of the pixel side in degrees.
    :param center_lat: is the latitude of the center of the pixel. This value
    +/- half the `pixel-size` must not exceed 90/-90 degrees latitude or an
    invalid area will be calculated.
    :return: the rel area in hectares of a square pixel of side length
    `pixel_size` whose center is at latitude `center_lat`.
    """

    a = 6378137  # meters
    b = 6356752.3142  # meters
    e = math.sqrt(1 - (b/a)**2)
    area_list = []
    for f in [center_lat+pixel_size/2, center_lat-pixel_size/2]:
        zm = 1 - e*math.sin(math.radians(f))
        zp = 1 + e*math.sin(math.radians(f))
        area_list.append(
            math.pi * b**2 * (
                math.log(zp/zm) / (2*e) +
                math.sin(math.radians(f)) / (zp*zm)))
    return (pixel_size / 360. * (area_list[0] - area_list[1])) * np.power(10.0,-4)

def aggregate_one_region(out_image, out_transform, pixel_size, width_0, height_0, width_1, height_1):
    """
    aggregate_one_region performs the aggregation of the density observable in a
    given region supplied in the form of a masked raster of the observable.

    :param out_image: the masked raster layer to aggregate.
    :param out_transform: the Affine of the raster layer.
    :param pixel_size: the side length in degrees of each square raster tile.
    :param width_0: the starting value position of the width array
    :param height_0: the starting value position of the height array
    :param width_1: the end value position of the width array
    :param height_1: the end value position of the height array
    :return: the aggregated value of the observable in the specified region.
    """

    # Create a matrix of coordinates based on tile number.
    cols, rows = np.meshgrid(np.arange(width_0, width_1), np.arange(height_0, height_1))

    # Transform the tile number coordinates to real coordinates and extract only
    # latitude information.
    ys = rasterio.transform.xy(out_transform, rows, cols)[1] # [0] is xs
    latitudes = np.array(ys) # Cast the list of arrays to a 2D array for computational convenience.
    ys = cols = rows = None #empty the memory

    # Iterate over the latitudes matrix, calculate the area of each tile, and
    # store it in the real_raster_areas array.
    real_raster_areas = np.empty(np.shape(latitudes))
    for i, latitude_array in enumerate(latitudes):
        for j, latitude in enumerate(latitude_array):
            real_raster_areas[i,j] = area_of_pixel(pixel_size, latitude)

    # Calculate the total value in each tile: density * area = observable value
    # in the area.
    value = real_raster_areas * out_image[0,height_0:height_1,width_0:width_1] #I don't think np.transpose() is necesary
    out_image = None #empty the memory
    # Sum all the carbon stock values in the country treating NaNs as 0.0.
    aggregated_value = np.nansum(value)

    return aggregated_value

def aggregate_one_region_dask(out_image, out_transform, pixel_size, width_0, height_0, width_1, height_1):

    # ---- 1. Compute one latitude per row ----
    # Create an array of row indices for the selected window.
    rows = np.arange(height_0, height_1)
    cols = np.full(rows.shape, width_0)

    # rasterio.transform.xy returns (xs, ys) for each (row, col) pair.
    # We only extract the latitude = ys.
    # ys is a list/array of latitudes for each row.
    row_lats = np.array(rasterio.transform.xy(out_transform, rows, cols)[1])

    # ---- 2. Compute area per row ----
    # Since each row has a constant latitude, we compute ONE area per row:
    row_areas = np.array([area_of_pixel(pixel_size, lat) for lat in row_lats])

    # ---- 3. Broadcast using Dask ----

    # row_areas has shape: (num_rows,)
    # We want to broadcast it against the raster slice (num_rows, num_cols).
    # So we reshape it to (num_rows, 1) using [:, None].
    # We transform the array [1, 2, 3] → [[1], (column vector)
    #                                     [2],
    #                                     [3]]
    area_matrix = da.from_array(row_areas[:, None], chunks=(512, 1))

    density = da.from_array(
                out_image[0, # band of the image
                height_0:height_1,
                width_0:width_1],
                chunks=(512, 512) # splits the image into 512×512 blocks
                )

    # ---- 4. Multiply + sum ----
    values = density * area_matrix
    sum_result = da.nansum(values).compute() # sum everything, treating NaN as 0
    return sum_result


def aggregate_density_observable(raster_files_list, region_polygons, temp_export_path):
    """
    aggregate_density_observable aggregates the density observable for all the
    raster files specified and inside the specified regions. The result of the
    aggregation is returned as a table and the result for each year/raster is
    progressively exported in CSV format.

    :param raster_files_list: a list containing the addresses of all the raster
    files that store the observable's data for each year.
    :param region_polygons: a GeoDataFrame storing the polygons corresponding to
    each region used for the aggregation.
    :param temp_export_path: path to export the temporary results.
    :return: a DataFrame storing the aggregated vegetation carbon stocks at the
    region level for each year.
    """

    # Final DataFrame will store the aggregated carbon stocks for each country and each year.
    aggregated_df = pd.DataFrame([])

    for file in raster_files_list[:]: # [10:]
        # Get the string values as a list. 
        # file_year = re.findall(r'\d+', file)[3]
        file_year = file.split('_')[-2]
        try:
        #    landcover_class = re.findall(r'\d+', file)[4]
           landcover_class = file.split('_')[-1].replace(".tif", "")
        except:
            landcover_class = False
            
        print("Processing file {} corresponding to year {}.".format(file, file_year))

        # This list will store the results from the aggregation.
        aggregated_value_list = []

        with rasterio.open(file) as raster_file: # Load the raster file.

            gt = raster_file.transform # Get all the raster properties on a list.
            pixel_size = gt[0] # X size is stored in position 0, Y size is stored in position 4.

            error_countries_id = [] # Create a list for all encountered possible errors
            
            # create both counter if we want to tmp export resuls at reaching n number of countries
            # country_counter = 1
            # country_counter_iterator = 1

            # Take the geometry name
            geometry_name = region_polygons.geometry.name

            for row_index, row in region_polygons.iterrows(): # gdf.loc[0:1].iterrows(): / gdf.loc(axis=0)[0:1] / df[df['column'].isin([1,2])]
                total_aggregated_value = 0
                try:
                    # Iterate over the country polygons to progressively calculate the total carbon stock in each one of them.

                    geodf_row = gpd.GeoDataFrame(geometry=gpd.GeoSeries(row[geometry_name]), crs=4326) # This is the country's polygon geometry df.
                    # geo_row = gpd.GeoSeries(row['geometry']) # This is the country's polygon geometry.

                    # create a grid of the territory with the coresponding EPSG
                    grid = get_geometry_grid(geodf_row, "EPSG:4326")
                    print("grid memory usage {}".format(sys.getsizeof(grid)))
                    # adjust the grid to the shape of the territory
                    region_grid = grid.overlay(geodf_row, how="intersection").to_crs(epsg='4326') # this operation requires both inputs to be gdf.
                    print("region_grid memory usage {}".format(sys.getsizeof(region_grid)))
                    # region_grid.to_file('region_grid.shp') # check memory for testing
                    # iterate over each tile and accumulate the value
                    
                    total_tiles = int(len(region_grid))
                    for row_index, tile_row in region_grid.iterrows():
                        # print a process index.
                        print("{} out of {} of country {}".format(row_index + 1, total_tiles, row["gid_n"]))
                        geo_tile_row = gpd.GeoSeries(tile_row['geometry'])
                        # repeat the masking process with the tile.
                        try:
                            # If it does not overlap
                            out_image, out_transform = rasterio.mask.mask(raster_file, geo_tile_row, crop=True, filled=True, nodata=np.nan) # make sure the nodata goes to nan.
                        # print("out_image memory usage {}".format(out_image.nbytes / np.power(10.0,9))) # check memory for testing
                        except Exception as e:
                            # print(f"The tile {tile_row} has errors: ", e)
                            continue

                        if np.isnan(out_image).all():
                            continue

                        # Obtain the number of tiles in both directions.
                        height = out_image.shape[1]
                        width  = out_image.shape[2]
                        #calculate the aggregated value.
                        # aggregated_value = aggregate_one_region(out_image, out_transform, pixel_size, 0, 0, width, height)

                        aggregated_value = aggregate_one_region_dask(out_image, out_transform, pixel_size, 0, 0, width, height)

                        # print("aggregated value memory usage {}".format(sys.getsizeof(aggregated_value))) # check memory for testing
                        # accumulate the results.
                        total_aggregated_value += aggregated_value 
                        # clean memory
                        out_image = None # this cleans the memory to 16 bits
                        out_transform = None

                        #force cleaning the memory
                        gc.collect()
                        
                except Exception as e:
                    # In case there is an error on the process, a value of -9999.0 will be appended
                    print("the country {} with index {} has errors: {}".format(row["gid_n"], row["id"], e) )
                    error_countries_id.append(row["id"])
                    aggregated_value = -9999.0 

                
                # Add the aggregated stock to the list.
                aggregated_value_list.append(total_aggregated_value)

                print("the country {} with index {} is finished with total carbon of: {}".format(row["gid_n"], row["id"], total_aggregated_value))
                
                """this part is to create additional internal temporal results"""
                # country_counter += 1
                # if country_counter_iterator < 26: # + 1
                #     country_counter_iterator += 1
                # else:
                # if not landcover_class:
                #     aggregated_observable = pd.DataFrame(row["ADM0_NAME"], aggregated_value, columns = ["country", file_year])
                #     print(aggregated_observable.head())
                #     # Export the temporary results from curent year.
                #     aggregated_observable.to_csv(temp_export_path + "_" + row["ADM0_NAME"] + "_" + str(file_year) + ".csv")
                # else:
                #     aggregated_observable = pd.DataFrame(row["ADM0_NAME"], aggregated_value, columns = ["country", file_year + "_" + landcover_class])
                #     aggregated_observable.to_csv(temp_export_path + "_" + row["ADM0_NAME"] + "_" + str(file_year) + "_" + str(landcover_class) + ".csv")

                    # country_counter_iterator = 1

        print("Finished calculating {}.".format(file_year))
        if error_countries_id:
            print("countries id with error: ", error_countries_id)

        # Transform the list to a DataFrame using the year as header.
        if not landcover_class:
            aggregated_observable = pd.DataFrame(aggregated_value_list, columns = [file_year])
            # Export the temporary results from curent year.
            aggregated_observable.to_csv(temp_export_path + "_" + str(file_year) + ".csv")
        else:
            aggregated_observable = pd.DataFrame(aggregated_value_list, columns = [file_year + "_" + landcover_class])
            # aggregated_observable.to_csv(os.path.join(temp_export_path, row["gid_n"] + "_" + str(file_year) + "_" + str(landcover_class) + ".csv"))
            aggregated_observable.to_csv(os.path.join(temp_export_path, os.path.basename(file).replace(".tif","") + ".csv"))
            

        # Merge this year's results with the final, multi-year DataFrame.
        aggregated_df = pd.merge(aggregated_df, aggregated_observable, how='outer', left_index = True, right_index=True)

    aggregated_df.to_csv(os.path.join(temp_export_path, row["gid_n"] + "_total_" + str(file_year) + ".csv"))

    return aggregated_df


In [ ]:
"""Load the input"""
gdf_gadm = load_vector_layer(
    db_name='',
    user='',
    password='',
    host='192.168.250.100',
    port=5555,
    table_name='administrative_units_un_gadm_level0',
    schema='public'
)

Successfully loaded administrative_units_un_gadm_level0 (318 features)


In [5]:
"""Filter the country"""
countries = ["Uganda"]
gdf_gadm_countries = filter_countries(gdf_gadm, countries)
# Export it to load it instantly in the process
gdf_gadm_countries.to_file("uganda_gadm_countries.shp")

In [4]:
"""Load the country"""
gdf_gadm_countries = gpd.read_file("Y:\z_resources\im-data-global-landcover\lc_analysis\output_fin_boundary.shp")
gdf_gadm_countries.head()

,location,gid_n,id,geometry
0,finland_glc_fcs30d_landcover_epsg4326_30m_2020...,Finland,1,"POLYGON ((23.20738 61.24803, 23.20738 61.74748..."


In [5]:
raster_files_path = r"Y:\z_resources\im-data-global-landcover\lc_analysis\finland_epsg4326__30m_2020"
raster_files_list = get_raster_file_list(raster_files_path)
temp_export_path = r"Y:\z_resources\im-data-global-landcover\lc_analysis\output_tmp"
aggregate_density_observable(raster_files_list, gdf_gadm_countries, temp_export_path)

Processing file Y:\z_resources\im-data-global-landcover\lc_analysis\finland_epsg4326__30m_2020\finland_epsg4326__30m_2020.tiff corresponding to year 30m.
grid memory usage 152
region_grid memory usage 152
1 out of 1 of country Finland
the country Finland with index 1 is finished with total carbon of: 0
Finished calculating 30m.


,30m_2020f
0,0
